# COVER-KBC Profile F1 Full TEST Runner

This notebook runs the committed COVER-KBC Profile F1 system end to end on the official TEST split:

1. Clone or update the repository.
2. Install the neural runtime dependencies.
3. Load the single frozen Mistral checkpoint declared by the config.
4. Run `scripts/run_cover.py` over all TEST rows.
5. Validate and package the produced `predictions.jsonl`.

No prior TEST prediction artifact is used as input. The output is generated by the full system runner from the committed config.

In [ ]:
# CELL 0 - Runtime paths
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/vquclinh/FactElicit-AKBC"
REPO = Path("/content/FactElicit-AKBC")
CONFIG_PATH = REPO / "configs/experiments/cover_kbc_v3_8_profile_f1_stock_empty_rescue_test.yaml"
TEST_PATH = REPO / "benchmark/data/test.jsonl"
RUN_DIR = Path("/content/cover_kbc_profile_f1_full_test")
PREDICTIONS = RUN_DIR / "predictions.jsonl"
SUBMISSION_ZIP = RUN_DIR / "submission.zip"
DRIVE_OUT = Path("/content/drive/MyDrive/cover_kbc/profile_f1_full_test")

print("Repo:", REPO)
print("Config:", CONFIG_PATH)
print("TEST split:", TEST_PATH)
print("Run output:", RUN_DIR)
print("Submission zip:", SUBMISSION_ZIP)
print("Drive output directory:", DRIVE_OUT)

In [ ]:
# CELL 0B - Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("Drive output directory:", DRIVE_OUT)

In [ ]:
# CELL 1 - Clone or pull repository code
import os
import subprocess
from pathlib import Path

if REPO.exists():
    os.chdir(REPO)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    os.chdir(REPO)

print("cwd:", Path.cwd())
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["git", "status", "--short"], check=True)

In [ ]:
# CELL 2 - Install package and model dependencies
import os
import subprocess
import sys

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[hf]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "mistral-common>=1.6.2"], check=True)

src_path = str(REPO / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import cover_kbc
print("cover_kbc:", cover_kbc.__file__)

In [ ]:
# CELL 3 - Hugging Face login
import getpass
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as exc:
    print("Colab secret lookup failed; falling back to env/manual token:", repr(exc))
    token = None

token = token or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token:
    token = getpass.getpass("Enter HF_TOKEN: ")

assert token and token.strip(), "HF_TOKEN is required to load the gated model."
os.environ["HF_TOKEN"] = token.strip()
os.environ["HUGGING_FACE_HUB_TOKEN"] = token.strip()
login(token=token.strip(), add_to_git_credential=False)
print("HF login completed.")

In [ ]:
# CELL 4 - Static preflight
import json
from collections import Counter
from pathlib import Path
import subprocess
import sys
import yaml

os.chdir(REPO)
assert CONFIG_PATH.exists(), f"Missing config: {CONFIG_PATH}"
assert TEST_PATH.exists(), f"Missing official TEST split: {TEST_PATH}"

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

cfg = yaml.safe_load(CONFIG_PATH.read_text())
assert cfg["experiment"]["name"] == "cover_kbc_v3_8_profile_f1_stock_empty_rescue_test"
assert cfg["relation_refinement"]["enabled"] is True
assert cfg["relation_refinement"]["features"]["mistral_stock_empty_rescue"] is True
assert "Qwen" not in json.dumps(cfg.get("model_profile", {}), sort_keys=True)

test_rows = read_jsonl(TEST_PATH)
assert len(test_rows) == 475, f"TEST split must have 475 rows, found {len(test_rows)}"
relation_counts = Counter(row["Relation"] for row in test_rows)

print("Config OK:", CONFIG_PATH)
print("TEST rows:", len(test_rows))
print("Relation counts:", dict(sorted(relation_counts.items())))
subprocess.run([sys.executable, "scripts/audit_model_budget.py", str(CONFIG_PATH)], check=True)

In [ ]:
# CELL 5 - Run full TEST inference
import shutil
import subprocess
import sys

os.chdir(REPO)
if RUN_DIR.exists():
    raise FileExistsError(f"{RUN_DIR} already exists. Move/delete it before starting a fresh full run.")

cmd = [
    sys.executable,
    "-u",
    "scripts/run_cover.py",
    "--config",
    str(CONFIG_PATH),
    "--split",
    "test",
    "--output-dir",
    str(RUN_DIR),
    "--no-eval",
]
print("Running:", " ".join(cmd), flush=True)
log_path = RUN_DIR.with_suffix(".log")
print("Log path:", log_path, flush=True)
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0,
    )
    assert process.stdout is not None
    while True:
        chunk = process.stdout.read(1)
        if chunk == "" and process.poll() is not None:
            break
        if not chunk:
            continue
        print(chunk, end="", flush=True)
        log.write(chunk)
        log.flush()
    return_code = process.wait()
print("Return code:", return_code)
if return_code != 0:
    raise SystemExit(return_code)
assert PREDICTIONS.exists(), f"Full runner did not write {PREDICTIONS}"

In [ ]:
# CELL 6 - Validate and package predictions
import hashlib
import json
from collections import Counter
import subprocess
import sys

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

pred_rows = read_jsonl(PREDICTIONS)
test_rows = read_jsonl(TEST_PATH)
assert len(pred_rows) == len(test_rows) == 475
assert [(r["SubjectEntity"], r["Relation"]) for r in pred_rows] == [
    (r["SubjectEntity"], r["Relation"]) for r in test_rows
]

relation_counts = Counter(row["Relation"] for row in pred_rows)
empty_by_relation = {
    relation: sum(not row.get("ObjectEntities") for row in pred_rows if row["Relation"] == relation)
    for relation in sorted(relation_counts)
}
print("Prediction rows:", len(pred_rows))
print("Prediction sha256:", sha256_file(PREDICTIONS))
print("Relation counts:", dict(sorted(relation_counts.items())))
print("Empty rows by relation:", empty_by_relation)

cmd = [
    sys.executable,
    "scripts/package_submission.py",
    "--predictions",
    str(PREDICTIONS),
    "--input",
    str(TEST_PATH),
    "--split",
    "test",
    "--out",
    str(SUBMISSION_ZIP),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Submission zip:", SUBMISSION_ZIP)
print("Zip sha256:", sha256_file(SUBMISSION_ZIP))

In [ ]:
# CELL 7 - Inspect run artifacts
import json

for name in [
    "manifest.json",
    "run_accounting.json",
    "relation_refinement.jsonl",
    "refinement_accounting.json",
    "calls.jsonl",
]:
    path = RUN_DIR / name
    print(name, "present" if path.exists() else "missing", path)

accounting_path = RUN_DIR / "run_accounting.json"
if accounting_path.exists():
    accounting = json.loads(accounting_path.read_text())
    print(json.dumps(accounting, indent=2, sort_keys=True)[:4000])

In [ ]:
# CELL 8 - Save artifacts to Google Drive
import json
import shutil

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
if DRIVE_OUT.exists():
    for child in DRIVE_OUT.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

DRIVE_RUN_DIR = DRIVE_OUT / RUN_DIR.name
shutil.copytree(RUN_DIR, DRIVE_RUN_DIR)

summary = {
    "config": str(CONFIG_PATH),
    "test_split": str(TEST_PATH),
    "run_dir": str(RUN_DIR),
    "predictions": str(PREDICTIONS),
    "predictions_sha256": sha256_file(PREDICTIONS),
    "submission_zip": str(SUBMISSION_ZIP),
    "submission_zip_sha256": sha256_file(SUBMISSION_ZIP),
    "drive_dir": str(DRIVE_OUT),
}
(DRIVE_OUT / "profile_f1_full_test_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("Saved run directory to:", DRIVE_RUN_DIR)
print("Saved summary to:", DRIVE_OUT / "profile_f1_full_test_summary.json")

In [ ]:
# CELL 9 - Download artifacts from the browser session
from google.colab import files

files.download(str(SUBMISSION_ZIP))
files.download(str(PREDICTIONS))
manifest_path = RUN_DIR / "manifest.json"
if manifest_path.exists():
    files.download(str(manifest_path))

In [ ]:
# CELL 10 - Disconnect runtime after artifacts are saved
from google.colab import runtime

runtime.unassign()